In [17]:
suppressPackageStartupMessages({
    require(circlize)
    library(dplyr)
})

In [28]:
mat  <- read.delim("Brain_chordates_high_res/Brain_chordates_Refined_family.MappingTables.links.csv", header = T, sep = ",")

In [29]:
mat$groups1 <- vapply(mat$source, FUN = function(x){strsplit(x, split = '_')[[1]][2]}, FUN.VALUE = character(1))
mat$groups2 <- vapply(mat$target, FUN = function(x){strsplit(x, split = '_')[[1]][2]}, FUN.VALUE = character(1))

In [30]:
mat[mat$source == 'hs_Diencephalon GABAergic neurons', ]

,source,target,value,groups1,groups2
,<chr>,<chr>,<dbl>,<chr>,<chr>
12,hs_Diencephalon GABAergic neurons,mm_Mesencephalon GABAergic neurons,0.6337466,Diencephalon GABAergic neurons,Mesencephalon GABAergic neurons
13,hs_Diencephalon GABAergic neurons,mm_Mesencephalon cholinergic neurons,0.1801335,Diencephalon GABAergic neurons,Mesencephalon cholinergic neurons
14,hs_Diencephalon GABAergic neurons,mm_Mesencephalon glutamatergic neurons,0.1056547,Diencephalon GABAergic neurons,Mesencephalon glutamatergic neurons
15,hs_Diencephalon GABAergic neurons,mm_Peptidergic neurons,0.3463462,Diencephalon GABAergic neurons,Peptidergic neurons
16,hs_Diencephalon GABAergic neurons,mm_Rhombencephalon cholinergic neurons,0.1310203,Diencephalon GABAergic neurons,Rhombencephalon cholinergic neurons
17,hs_Diencephalon GABAergic neurons,pv_Mesencephalon GABAergic neurons,0.6105214,Diencephalon GABAergic neurons,Mesencephalon GABAergic neurons
18,hs_Diencephalon GABAergic neurons,pm_Diencephalon GABAergic neurons,0.2558659,Diencephalon GABAergic neurons,Diencephalon GABAergic neurons
19,hs_Diencephalon GABAergic neurons,pm_Dopaminergic neurons,0.1071346,Diencephalon GABAergic neurons,Dopaminergic neurons
20,hs_Diencephalon GABAergic neurons,pm_Mesencephalon GABAergic neurons,0.6174078,Diencephalon GABAergic neurons,Mesencephalon GABAergic neurons


In [31]:
# keep only neuronal major groups
n <- c('Diencephalon glutamatergic neurons', 'Diencephalon GABAergic neurons','Mesencephalon glutamatergic neurons',
      'Mesencephalon GABAergic neurons','Rhombencephalon glutamatergic neurons','Rhombencephalon GABAergic neurons',
      'Splatter', 'Telencephalon glutamatergic neurons', 'Telencephalon GABAergic neurons',
       18,11,10,13,7,15,0,1,14,21,2,17,16,9)

mat <- mat %>% filter(groups1 %in% n & groups2 %in% n)
mat <- mat %>% mutate(groups1 = factor(groups1, levels = rev(n)), 
                     groups2 = factor(groups2, levels = rev(n))) %>%
    arrange(groups1)

In [32]:
color = setNames(n, c('#EEBEC0','#23B2E0','#BD9E64','#6483A4','#FAAE5F',
                      '#73ABCF','#000000','#EF2C2B','#27447C',
                      "#0072B2", "#E69F00", "#56B4E9", "#56B4E9", "#009E73", "#D55E00","#F0E442","#F0E442",
                "#CC79A7", "#787878", "#663300", "#663300", "#008080", "#008080"))
color

#EEBEC0                                 #23B2E0 
   "Diencephalon glutamatergic neurons"        "Diencephalon GABAergic neurons" 
                                #BD9E64                                 #6483A4 
  "Mesencephalon glutamatergic neurons"       "Mesencephalon GABAergic neurons" 
                                #FAAE5F                                 #73ABCF 
"Rhombencephalon glutamatergic neurons"     "Rhombencephalon GABAergic neurons" 
                                #000000                                 #EF2C2B 
                             "Splatter"   "Telencephalon glutamatergic neurons" 
                                #27447C                                 #0072B2 
      "Telencephalon GABAergic neurons"                                    "18" 
                                #E69F00                                 #56B4E9 
                                   "11"                                    "10" 
                                #56B4E9                                 #009E73 
                                   "13"                                     "7" 
                                #D55E00                                 #F0E442 
                                   "15"                                     "0" 
                                #F0E442                                 #CC79A7 
                                    "1"                                    "14" 
                                #787878                                 #663300 
                                   "21"                                     "2" 
                                #663300                                 #008080 
                                   "17"                                    "16" 
                                #008080 
                                    "9"

In [33]:
mat$color <- names(color)[match(mat$groups1, color)]
mat$sp <- vapply(mat$source, FUN = function(x){
    strsplit(x, split = '_')[[1]][1]
}, FUN.VALUE = character(1))
mat$sp2 <- vapply(mat$target, FUN = function(x){
    strsplit(x, split = '_')[[1]][1]
}, FUN.VALUE = character(1))

In [34]:
# select linkages between amphioxus and vertebrates
mat <- mat %>% filter(sp == 'bf' | sp2 == 'bf')
# show only one direction
mat[mat$sp2 == 'bf', 'value'] = 0

In [35]:
# manually add gaps
test = vapply(unique(mat[[1]]), FUN = function(x){strsplit(x, split = '_')[[1]][1]},FUN.VALUE = character(1))
gaps = numeric(length(test))
for (i in 1:(length(gaps)-1)) {
  # Check if the current element is different from the previous one
  if (test[i] != test[i + 1]) {
    gaps[i] <- 1
  } else {
    gaps[i] <- 0
  }
}

In [36]:
circos.par(gap.after = gaps, track.margin = c(0, 0.01))
grid.col = setNames(mat$color, mat$source)
# highlight bf glia linkages
border_df = mat[grepl("bf", mat$source), 1:2]
border_df = rbind(border_df, mat[grepl("bf", mat$target), 1:2])
#border_df <- rbind(border_df, border_df2)
border_df$p = rep(x = 1, times = nrow(border_df))

In [37]:
pdf("vertebrate_neurons.chord_plot2.pdf", width = 20, height = 20)

species_colors <- c("hs" = "#989A9C", "mm" = "#F7D08D", "pv" = "#BF83A5", "pm" = "#8684B0", "bf" = '#702963')
chordDiagram(mat[,1:3], directional = 1, direction.type = c("arrows"), link.arr.type = "big.arrow", 
             grid.col = grid.col, link.visible = mat[[3]] > 0.2,
            annotationTrack = "grid", preAllocateTracks = list(track.height = 0.02))

circos.trackPlotRegion(
    track.index = 1,
  ylim = c(0, 1),  # Define the vertical range for the track
  track.height = 0.02,  # Adjust track height as needed
  bg.border = NA,  # No border for the track
  panel.fun = function(x, y) {
    sector.index <- CELL_META$sector.index  # Current sector
    species <- mat$sp[mat$source == sector.index]  # Match species to the current sector
    
    # Draw colored rectangles for each species
    circos.rect(
      xleft = CELL_META$cell.xlim[1], xright = CELL_META$cell.xlim[2], 
      ybottom = 0, ytop = 1, 
      col = species_colors[species], border = NA
    )
  }
)
circos.track(track.index = 2, panel.fun = function(x, y) {
    circos.text(CELL_META$xcenter, CELL_META$ylim[1], CELL_META$sector.index, 
        facing = "clockwise", niceFacing = TRUE, adj = c(0, 0.05), cex = 1)
}, bg.border = NA) # here set bg.border to NA is important


dev.off()
circos.clear()

pdf 
  2

In [16]:
# print some statistics

In [17]:
mat %>% filter(groups1 != groups2) %>% filter(sp =='pm' & groups2 %in% c('Oligodendrocytes', 'Oligodendrocyte precursor cells'))

source,target,value,groups1,groups2,color,sp
<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
pm_Ast_254,hs_OPC_556,0.2990799,Astrocytes,Oligodendrocyte precursor cells,#8BCF02,pm
pm_Epen_304,hs_OPC_556,0.2588697,Ependymal cells,Oligodendrocyte precursor cells,#B4DEA2,pm
pm_Erythrocytes_301,hs_Oligo_521,0.4456225,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,hs_Oligo_529,0.5084667,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,hs_Oligo_533,0.3069232,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,hs_Oligo_550,0.2966744,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,mm_Oligo_361,0.3543942,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,mm_Oligo_364,0.2323280,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
pm_Erythrocytes_301,mm_Oligo_365,0.3377884,Erythrocytes,Oligodendrocytes,#EF2C2B,pm


In [18]:
dim(mat)

[1] 1022    7

In [19]:
tmp <- mat %>% filter(groups1 != groups2) %>% 
    filter(!(groups1 == 'Astrocytes' & groups2 == 'Ependymal cells')) %>%
    filter(!(groups2 == 'Astrocytes' & groups1 == 'Ependymal cells')) %>% 
    #filter(!(groups1 == 'Fibroblasts' & groups2 == 'Vascular cells')) %>%
    #filter(!(groups2 == 'Fibroblasts' & groups1 == 'Vascular cells')) %>% 
    filter(!(groups1 == 'Oligodendrocytes' & groups2 == 'Oligodendrocyte precursor cells')) %>%
    filter(!(groups1 == 'Oligodendrocyte precursor cells' & groups2 == 'Oligodendrocytes')) 
# bidirectional linkages
tmp[c("source", "target")] <- t(apply(tmp[c("source", "target")], 1, sort))
tmp <- tmp[!duplicated(tmp[c("source", "target")]), ]
tmp

,source,target,value,groups1,groups2,color,sp
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,hs_OPC_556,pm_Ast_254,0.2990799,Astrocytes,Oligodendrocyte precursor cells,#8BCF02,pm
2,hs_OPC_556,pm_Epen_304,0.2588697,Ependymal cells,Oligodendrocyte precursor cells,#B4DEA2,pm
3,hs_Oligo_521,pm_Erythrocytes_301,0.4456225,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
4,hs_Oligo_529,pm_Erythrocytes_301,0.5084667,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
5,hs_Oligo_533,pm_Erythrocytes_301,0.3069232,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
6,hs_Oligo_550,pm_Erythrocytes_301,0.2966744,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
7,mm_Oligo_361,pm_Erythrocytes_301,0.3543942,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
8,mm_Oligo_364,pm_Erythrocytes_301,0.2323280,Erythrocytes,Oligodendrocytes,#EF2C2B,pm
9,mm_Oligo_365,pm_Erythrocytes_301,0.3377884,Erythrocytes,Oligodendrocytes,#EF2C2B,pm


In [20]:
mat %>% group_by(groups1) %>% count(sp, sort = TRUE) %>% arrange(groups1)

groups1,sp,n
<chr>,<chr>,<int>
Astrocytes,pv,90
Astrocytes,hs,81
Astrocytes,mm,77
Astrocytes,pm,52
Choroid plexus epithelial cells,hs,6
Choroid plexus epithelial cells,mm,6
Choroid plexus epithelial cells,pm,6
Choroid plexus epithelial cells,pv,2
Ependymal cells,pm,31


In [22]:
mat[c("source", "target")] <- t(apply(mat[c("source", "target")], 1, sort))
mat <- mat[!duplicated(mat[c("source", "target")]), ]
mat

,source,target,value,groups1,groups2,color,sp
,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,hs_Ast_593,mm_Ast_495,0.3765032,Astrocytes,Astrocytes,#8BCF02,hs
2,hs_Ast_593,mm_Ast_504,0.5169962,Astrocytes,Astrocytes,#8BCF02,hs
3,hs_Ast_593,mm_Ast_505,0.3592002,Astrocytes,Astrocytes,#8BCF02,hs
4,hs_Ast_593,mm_Ast_506,0.6811880,Astrocytes,Astrocytes,#8BCF02,hs
5,hs_Ast_593,mm_Ast_507,0.7968926,Astrocytes,Astrocytes,#8BCF02,hs
6,hs_Ast_593,mm_Ast_509,0.7639273,Astrocytes,Astrocytes,#8BCF02,hs
7,hs_Ast_593,pv_Ast_315,0.2301026,Astrocytes,Astrocytes,#8BCF02,hs
8,hs_Ast_593,pv_Ast_319,0.2218708,Astrocytes,Astrocytes,#8BCF02,hs
9,hs_Ast_593,pv_Ast_321,0.2078996,Astrocytes,Astrocytes,#8BCF02,hs


In [23]:
# ratio of mapping with the same families
(511-44)/511

[1] 0.9138943

In [20]:
?chordDiagram